# Clase 1 · Práctica 02 — Caso aplicado: analizando transacciones

### De los patrones a un problema real de negocio

Hoy aprendimos patrones algorítmicos (acumulador, contador, campeón, búsqueda).
Ahora los usamos sobre **datos reales** de una tienda con sucursales en varias
ciudades de Colombia.

> 🎯 **Importante:** todavía **no** usaremos NumPy ni pandas (vienen en las clases
> 5 y 6). Lo haremos "a mano", con listas y bucles, para que entiendas qué hacen
> esas librerías por dentro. Al final compararás cuánto código te ahorrarán.

**Preguntas de negocio que responderemos:**
1. ¿Cuánto vendió la tienda en total y cuál fue el ticket promedio?
2. ¿Cuál fue la transacción más alta? ¿Y la más baja?
3. ¿Qué ciudad generó más ingresos?
4. ¿Cuántas transacciones superaron los \$200.000 (posibles ventas mayoristas)?
5. ¿Hay valores atípicos (*outliers*) que debamos revisar?

In [ ]:
# Cargamos el dataset con la librería estándar 'csv' (sin pandas).
import csv, os, sys
sys.path.append(os.path.abspath(os.path.join("..", "shared")))
from verificador import revisar

ruta = os.path.join("..", "datasets", "transacciones.csv")
with open(ruta, encoding="utf-8") as f:
    filas = list(csv.DictReader(f))   # cada fila es un dict {columna: valor}

print(f"Se cargaron {len(filas)} transacciones.")
print("Primera fila:", filas[0])

### Paso 1 · Entender la estructura (Entrada)

Cada fila tiene: `id`, `fecha`, `ciudad`, `categoria`, `metodo_pago`, `monto`.

⚠️ Detalle crucial: al leer un CSV, **todo llega como texto** (`str`). El `monto`
`"73078"` es una cadena, no un número. Si intentáramos sumar cadenas obtendríamos
basura. Primer paso de toda limpieza de datos: **convertir los tipos**.

In [ ]:
# Extraemos los montos como ENTEROS (no como texto). Patrón: transformar la entrada.
montos = [int(fila["monto"]) for fila in filas]
print("Tipo del primer monto en el CSV:", type(filas[0]["monto"]).__name__)
print("Tipo tras convertir:            ", type(montos[0]).__name__)
print("Primeros 5 montos:", montos[:5])

### Pregunta 1 · Total e ingreso promedio (acumulador)

Usamos el patrón **acumulador** para el total, y lo dividimos por la cantidad para
el promedio. Exactamente lo que hicimos en la clase, ahora sobre 120 datos reales.

In [ ]:
total = 0
for m in montos:           # acumulador
    total += m
promedio = total / len(montos)

print(f"Ingreso total:   ${total:,.0f}")
print(f"Ticket promedio: ${promedio:,.0f}")
print(f"Transacciones:   {len(montos)}")

### Pregunta 2 · Transacción máxima y mínima (patrón campeón)

Dos campeones a la vez en un solo recorrido: uno que busca el mayor y otro el
menor. Recorrer una sola vez para obtener dos respuestas es más eficiente que
recorrer dos veces.

In [ ]:
maximo = montos[0]
minimo = montos[0]
for m in montos:
    if m > maximo:
        maximo = m
    if m < minimo:
        minimo = m

print(f"Transacción más alta: ${maximo:,.0f}")
print(f"Transacción más baja: ${minimo:,.0f}")
# Comprobamos con las funciones incorporadas:
print("¿Coincide max()?", maximo == max(montos), "| ¿min()?", minimo == min(montos))

### Pregunta 3 · ¿Qué ciudad vendió más? (agrupar sin diccionarios)

Agrupar por ciudad es un problema nuevo. Lo resolvemos con **dos listas
paralelas**: una con los nombres de ciudad y otra con su total acumulado. (En la
Clase 4 verás que un *diccionario* hace esto de forma más elegante; hoy lo hacemos
con las herramientas que ya dominas, para entender la mecánica.)

**Algoritmo:**
1. Por cada transacción, busca su ciudad en la lista de ciudades vistas.
2. Si ya está, súmale el monto en la posición correspondiente.
3. Si es nueva, agrégala con su monto inicial.

In [ ]:
ciudades = []     # nombres únicos de ciudad
totales = []      # total acumulado, alineado por posición con 'ciudades'

for fila in filas:
    ciudad = fila["ciudad"]
    monto = int(fila["monto"])
    # búsqueda lineal de la ciudad entre las ya vistas
    if ciudad in ciudades:
        i = ciudades.index(ciudad)   # posición de la ciudad
        totales[i] += monto
    else:
        ciudades.append(ciudad)
        totales.append(monto)

# Mostramos el resultado y encontramos la ciudad líder (otro 'campeón').
lider = ciudades[0]
mejor_total = totales[0]
for i in range(len(ciudades)):
    print(f"  {ciudades[i]:<14} ${totales[i]:>12,.0f}")
    if totales[i] > mejor_total:
        mejor_total = totales[i]
        lider = ciudades[i]

print(f"\n🏆 Ciudad con más ingresos: {lider} (${mejor_total:,.0f})")

### Pregunta 4 · Ventas mayoristas (contador con condición)

In [ ]:
UMBRAL = 200000
mayoristas = 0
for m in montos:
    if m > UMBRAL:
        mayoristas += 1
print(f"Transacciones por encima de ${UMBRAL:,}: {mayoristas}")
print(f"Eso es el {mayoristas / len(montos) * 100:.1f}% de las ventas.")

### Pregunta 5 · Detección de valores atípicos (*outliers*)

En el mundo real, los datos traen errores y rarezas. Una regla simple y común:
marcar como sospechoso todo monto que supere **3 veces el promedio**. No es una
verdad absoluta, pero es un buen primer filtro para *"esto merece una mirada
humana"*.

In [ ]:
limite = 3 * promedio
sospechosos = []
for fila in filas:
    if int(fila["monto"]) > limite:
        sospechosos.append(fila)

print(f"Umbral de sospecha (3x promedio): ${limite:,.0f}")
print(f"Transacciones atípicas encontradas: {len(sospechosos)}")
for s in sospechosos:
    print(f"  id {s['id']} | {s['ciudad']:<13} | ${int(s['monto']):>12,}")

> 🤔 **¿Qué pasaría si...?** elimináramos los outliers antes de calcular el
> promedio. ¿Subiría o bajaría el ticket promedio? Los valores atípicos altos
> *inflan* el promedio: por eso en estadística a veces se prefiere la **mediana**.
> Lo veremos con detalle más adelante.

---
## 🛠️ Tu turno

Ahora resuelve tú. Completa cada función y ejecuta su comprobación.

### Tarea A · Ingreso por método de pago

Completa `total_por_metodo(filas, metodo)`: la suma de los montos cuyas
transacciones usaron ese `metodo_pago`.

**Ejemplo:** `total_por_metodo(filas, "efectivo")` → suma de todas las compras en
efectivo.

In [ ]:
def total_por_metodo(filas, metodo):
    # ✏️ TU CÓDIGO AQUÍ (acumulador con condición)
    return None

# Comprobación: comparamos contra un cálculo de referencia hecho aparte.
_ref_efectivo = sum(int(f["monto"]) for f in filas if f["metodo_pago"] == "efectivo")
revisar("efectivo", total_por_metodo(filas, "efectivo") == _ref_efectivo)
_ref_tarjeta = sum(int(f["monto"]) for f in filas if f["metodo_pago"] == "tarjeta")
revisar("tarjeta", total_por_metodo(filas, "tarjeta") == _ref_tarjeta)

<details><summary>💡 Ver solución</summary>

```python
def total_por_metodo(filas, metodo):
    total = 0
    for f in filas:
        if f["metodo_pago"] == metodo:
            total += int(f["monto"])
    return total
```
</details>

### Tarea B · Contar transacciones de una ciudad

Completa `contar_ciudad(filas, ciudad)`: cuántas transacciones ocurrieron en esa
ciudad.

In [ ]:
def contar_ciudad(filas, ciudad):
    # ✏️ TU CÓDIGO AQUÍ (contador con condición)
    return None

_ref_bogota = sum(1 for f in filas if f["ciudad"] == "Bogota")
revisar("Bogota", contar_ciudad(filas, "Bogota") == _ref_bogota)
_ref_cali = sum(1 for f in filas if f["ciudad"] == "Cali")
revisar("Cali", contar_ciudad(filas, "Cali") == _ref_cali)

<details><summary>💡 Ver solución</summary>

```python
def contar_ciudad(filas, ciudad):
    cuenta = 0
    for f in filas:
        if f["ciudad"] == ciudad:
            cuenta += 1
    return cuenta
```
</details>

### Tarea C · Promedio de una categoría

Completa `promedio_categoria(filas, categoria)`: el promedio de los montos de esa
categoría. Si no hay ninguna transacción de esa categoría, devuelve `None`
(¡caso borde!).

In [ ]:
def promedio_categoria(filas, categoria):
    # ✏️ TU CÓDIGO AQUÍ (acumulador + contador + validar división)
    return None

_montos_tec = [int(f["monto"]) for f in filas if f["categoria"] == "tecnologia"]
_ref_tec = sum(_montos_tec) / len(_montos_tec)
revisar("tecnologia", promedio_categoria(filas, "tecnologia") == _ref_tec)
revisar("categoría inexistente -> None", promedio_categoria(filas, "no_existe") is None)

<details><summary>💡 Ver solución</summary>

```python
def promedio_categoria(filas, categoria):
    total = 0
    cuenta = 0
    for f in filas:
        if f["categoria"] == categoria:
            total += int(f["monto"])
            cuenta += 1
    if cuenta == 0:
        return None
    return total / cuenta
```
</details>

---
## Reflexión final: eficiencia y lo que viene

Fíjate en algo: para responder 5 preguntas recorrimos la lista de transacciones
**varias veces** (una por pregunta). Con 120 filas no importa, pero con 50 millones
de filas, cada recorrido cuesta. Parte del oficio es decidir *cuándo* combinar
trabajo en un solo recorrido y *cuándo* separar por claridad.

También notaste cuánto código hizo falta para agrupar por ciudad "a mano". En la
**Clase 6** harás exactamente esto con pandas en **una sola línea**:

```python
df.groupby("ciudad")["monto"].sum()
```

Pero ahora **sabes qué hace esa línea por dentro** — y esa comprensión es lo que
separa a quien usa una herramienta de quien la domina.

➡️ Sigue con **homework01.ipynb** y **homework02.ipynb** para afianzar lo aprendido.